# Noisy simulation and error mitigation
**Quantum Computing · Week 6 (Noise and real hardware) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/qc/w6d5.html

In [ ]:
# On Google Colab, this installs qiskit and qiskit-aer (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 qiskit-aer==0.17.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2 qiskit-aer==0.17.2

**Project: fight the noise.** Today's computers can't yet correct their errors, but we can
**mitigate** them: run a few extra, cleverly chosen circuits, and post-process. Two classic tools:
**readout-error mitigation** and **zero-noise extrapolation** (ZNE).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

noise = NoiseModel()
noise.add_all_qubit_quantum_error(depolarizing_error(0.002, 1), ["h", "x"])   # small one-qubit gate errors
noise.add_all_qubit_quantum_error(depolarizing_error(0.03, 2), ["cx"])        # 3% per CNOT
noise.add_all_qubit_readout_error(ReadoutError([[0.97, 0.03],                 # a 0 is read as 1 3% of the time
                                                [0.08, 0.92]]))               # a 1 is read as 0 8% of the time
sim = AerSimulator(noise_model=noise)

def run(circuit, shots=20000, seed=1):
    """Counts as probabilities for '00', '01', '10', '11' (no gate cancellation: optimization_level=0)."""
    counts = sim.run(transpile(circuit, sim, optimization_level=0), shots=shots, seed_simulator=seed).result().get_counts()
    return np.array([counts.get(format(i, "02b"), 0) / shots for i in range(4)])

def bell(folds=1):
    qc = QuantumCircuit(2)
    qc.h(0)
    for _ in range(folds):            # 1, 3 or 5 CNOTs: since CX * CX = I, the same circuit with more noise
        qc.cx(0, 1)
    qc.measure_all()
    return qc

zz = lambda p: p[0] - p[1] - p[2] + p[3]          # <ZZ> from the probabilities of 00, 01, 10, 11
p_noisy = run(bell())
print("probabilities of 00, 01, 10, 11:", p_noisy.round(3), "  ideal: [0.5 0 0 0.5]")
print(f"<ZZ> = {zz(p_noisy):.3f} (ideal: 1)")

## Readout-error mitigation
Prepare each of the four basis states, measure, and record how often each is read as what: the
**calibration matrix** $M$ (column = prepared, row = read). Noisy readout turns the true probabilities
into $M\,p$; multiplying by $M^{-1}$ undoes it.

In [ ]:
M = np.zeros((4, 4))
for i in range(4):
    prep = QuantumCircuit(2)
    for q, bit in enumerate(reversed(format(i, "02b"))):
        if bit == "1":
            prep.x(q)
    prep.measure_all()
    M[:, i] = run(prep, seed=10 + i)
print("calibration matrix (column = prepared 00, 01, 10, 11):")
print(M.round(3))

p_fixed = np.linalg.solve(M, p_noisy)              # the same as M^-1 @ p_noisy
print("after readout mitigation:", p_fixed.round(3), f"  <ZZ> = {zz(p_fixed):.3f}")

Much better, but not 1: the 3% CNOT error is still there. That's gate noise, and the calibration
can't see it.

## Zero-noise extrapolation
Make the gate noise **worse on purpose**: replace the CNOT by 3 or 5 CNOTs (the same operation,
since $CX \cdot CX = I$, but 3 or 5 times the noise). Measure at noise scales $\lambda = 1, 3, 5$, fit a
curve, and read it at $\lambda = 0$.

In [ ]:
lams = np.array([1, 3, 5])
values = np.array([zz(np.linalg.solve(M, run(bell(k), seed=k))) for k in lams])   # readout-mitigated
linear = np.polyfit(lams, values, 1)
richardson = np.polyfit(lams, values, 2)
print("<ZZ> at noise scale 1, 3, 5:", values.round(4))
print(f"straight-line extrapolation to 0: {np.polyval(linear, 0):.4f}")
print(f"parabola (Richardson) to 0:       {np.polyval(richardson, 0):.4f}")

xs = np.linspace(0, 5.5, 100)
plt.plot(lams, values, "o", color="#7b5cfa", ms=8, label="measured (readout-mitigated)")
plt.plot(xs, np.polyval(linear, xs), color="#b9a7ff", label="straight line")
plt.plot(xs, np.polyval(richardson, xs), color="#ff8a50", ls="--", label="parabola")
plt.axhline(1, color="#17a98c", ls=":", label="ideal")
plt.xlabel(r"noise scale $\lambda$")
plt.ylabel(r"$\langle ZZ\rangle$")
plt.legend()
plt.show()

## Project: put it all together
Compare four estimates of the Bell state's $\langle ZZ\rangle$ (ideal: 1), each from 20,000 shots per circuit:
1. **raw**: one run, no mitigation;
2. **readout only**: the calibration matrix, no extrapolation;
3. **ZNE only**: extrapolate the *raw* values at $\lambda = 1, 3, 5$ (straight line);
4. **both**: extrapolate the readout-mitigated values.

Which one gets closest? Why can't ZNE alone fix the readout errors?

In [ ]:
raw_values = np.array([zz(run(bell(k), seed=k)) for k in lams])
estimates = {
    "raw": raw_values[0],
    "readout only": 0.0,     # TODO: zz of the readout-mitigated probabilities at lambda = 1
    "ZNE only": 0.0,         # TODO: np.polyval(np.polyfit(lams, raw_values, 1), 0)
    "both": 0.0,             # TODO: the same, using the readout-mitigated values
}
for name, value in estimates.items():
    print(f"{name:13s} {value:.4f}   error {abs(value - 1):.4f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
raw_values = np.array([zz(run(bell(k), seed=k)) for k in lams])
fixed_values = np.array([zz(np.linalg.solve(M, run(bell(k), seed=k))) for k in lams])
estimates = {
    "raw": raw_values[0],
    "readout only": fixed_values[0],
    "ZNE only": np.polyval(np.polyfit(lams, raw_values, 1), 0),
    "both": np.polyval(np.polyfit(lams, fixed_values, 1), 0),
}
for name, value in estimates.items():
    print(f"{name:13s} {value:.4f}   error {abs(value - 1):.4f}")
print("Folding multiplies the gate noise but not the readout noise, so extrapolation alone can't remove it.")
print("Each tool fixes its own kind of error; together they land within a percent of the ideal, at the price of")
print("running 3 circuits plus 4 calibrations instead of 1.")